# Notebook 07 — Validation Harness

**Prerequisites:**
- Notebook 04 completed (docs ingested)
- Ollama running with models loaded
- Postgres running

**What to validate:**
- Load gold YAML → canned_questions table
- Run each question as Global Auditor (bypasses role filter)
- Cosine similarity scoring
- Pass/fail table
- Histogram of scores
- Prompt improvement loop demo

In [1]:
import sys
sys.path.insert(0, '..')

from sqlalchemy import select

from backend.db.session import get_session
from backend.db.models import User, Role, RoleType
from backend.llm.ollama_provider import OllamaProvider
from backend.rag.chromadb_provider import ChromaDBProvider
from backend.validation.runner import load_gold_standards, run_validation, print_validation_report
from backend.validation.scorer import score_answer, evaluate_batch, DEFAULT_PASS_THRESHOLD

llm = OllamaProvider()
rag = ChromaDBProvider(persist_dir='../data/chroma_db', collection_name='notebook_04_test')

print(f'ChromaDB chunks available: {rag.get_chunk_count()}')
print('Ready')

/Users/aarthi/code/policy_system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChromaDB chunks available: 2
Ready


## 1. Load Gold Standards

In [2]:
async def load_questions():
    async with get_session() as session:
        # Get or create the Global Auditor user for running QA
        auditor = (await session.execute(select(User).where(User.email == 'auditor@example.com'))).scalar_one_or_none()
        if auditor is None:
            print('No auditor user found — run notebook 01 first')
            return None, None
        
        questions = await load_gold_standards(
            session,
            created_by=auditor.user_id,
        )
        print(f'Loaded {len(questions)} canned questions:')
        for q in questions:
            print(f'  [{q.domain or "all"}] {q.question_text[:60]}...')
        return questions, auditor

questions, auditor_user = await load_questions()

Loaded 7 canned questions:
  [IT] What are the password requirements for employees?...
  [IT] What is required when accessing restricted data?...
  [IT] What should employees do if a security incident occurs?...
  [Finance] What is the expense reporting deadline?...
  [Finance] When do vendor invoices require two signatories?...
  [IT] What MFA method is required for system access?...
  [Finance] What happens if a department exceeds its budget by more than...


## 2. Run Validation Harness

In [3]:
if questions and auditor_user:
    async def run_all():
        async with get_session() as session:
            runs = await run_validation(
                session=session,
                rag_provider=rag,
                llm_provider=llm,
                runner_user=auditor_user,
                questions=questions,
                pass_threshold=DEFAULT_PASS_THRESHOLD,
            )
            return runs
    
    validation_runs = await run_all()
    print_validation_report(validation_runs, questions)


VALIDATION REPORT — 0/7 passed (0%)
Average similarity score: 0.000

✗ FAIL | Score=0.000 | Domain=IT
  Q: What are the password requirements for employees?
  AI:   No documents accessible for this question....
  Gold: Employees must use passwords that are at least 12 characters long. Passwords must be changed every 90 days. Shared crede...

✗ FAIL | Score=0.000 | Domain=IT
  Q: What is required when accessing restricted data?
  AI:   No documents accessible for this question....
  Gold: Restricted data must be encrypted both at rest and in transit. Access to Restricted data requires manager approval and a...

✗ FAIL | Score=0.000 | Domain=IT
  Q: What should employees do if a security incident occurs?
  AI:   No documents accessible for this question....
  Gold: Security incidents must be reported to the IT Security team within 1 hour of discovery. The incident response team will ...

✗ FAIL | Score=0.000 | Domain=Finance
  Q: What is the expense reporting deadline?
  AI:   No docume

## 3. Score Distribution Histogram

In [4]:
if questions and validation_runs:
    try:
        import matplotlib.pyplot as plt
        
        scores = [r.similarity_score for r in validation_runs]
        
        plt.figure(figsize=(8, 4))
        plt.hist(scores, bins=10, range=(0, 1), color='steelblue', edgecolor='white')
        plt.axvline(x=DEFAULT_PASS_THRESHOLD, color='red', linestyle='--', label=f'Pass threshold ({DEFAULT_PASS_THRESHOLD})')
        plt.xlabel('Cosine Similarity Score')
        plt.ylabel('Count')
        plt.title('Validation Score Distribution')
        plt.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('matplotlib not installed — skipping histogram')
        print(f'Scores: {[round(r.similarity_score, 3) for r in validation_runs]}')

matplotlib not installed — skipping histogram
Scores: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 4. Standalone Scorer Test

In [5]:
# Test the scorer without needing Ollama
print('Testing standalone cosine similarity scorer...')

# High similarity — same meaning, different words
score_high = score_answer(
    ai_answer='MFA is required for all employees accessing company systems.',
    gold_answer='All employees must use multi-factor authentication when accessing company systems.',
)
print(f'High similarity (MFA paraphrase): {score_high:.3f}')
assert score_high > 0.7, f'Expected > 0.7, got {score_high}'

# Low similarity — completely different topic
score_low = score_answer(
    ai_answer='The sky is blue and clouds are white.',
    gold_answer='Passwords must be changed every 90 days and be at least 12 characters.',
)
print(f'Low similarity (unrelated): {score_low:.3f}')
assert score_low < 0.5, f'Expected < 0.5, got {score_low}'

print('Scorer: PASS')

Testing standalone cosine similarity scorer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/103 [00:00<00:00, 17623.13it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/103 [00:00<00:00, 1028.02it/s, Materializing param=embeddings.LayerNorm.bias] 

Loading weights:   2%|▏         | 2/103 [00:00<00:00, 819.04it/s, Materializing param=embeddings.LayerNorm.weight]

...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1867.67it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1861.66it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1853.02it/s, Materializing param=pooler.dense.weight]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


High similarity (MFA paraphrase): 0.785
Low similarity (unrelated): -0.173
Scorer: PASS


## 5. Prompt Improvement Loop Demo

In [6]:
# Demo: compare two prompt variants for the same question
# In a real improvement loop, you'd modify prompts.py and re-run

gold = 'Passwords must be at least 12 characters and changed every 90 days.'

# Simulate two different AI responses (as if from different prompt variants)
response_v1 = 'Employee passwords should be long and changed regularly per company policy.'
response_v2 = 'The IT Security Policy requires passwords to be minimum 12 characters and rotated every 90 days.'

score_v1 = score_answer(response_v1, gold)
score_v2 = score_answer(response_v2, gold)

print('Prompt comparison:')
print(f'  Prompt v1 (vague):    score={score_v1:.3f}')
print(f'  Prompt v2 (specific): score={score_v2:.3f}')
print(f'  Winner: {"v2" if score_v2 > score_v1 else "v1"}')
print('\nUse this loop to compare prompts.py variants and pick the best one.')

Prompt comparison:
  Prompt v1 (vague):    score=0.695
  Prompt v2 (specific): score=0.779
  Winner: v2

Use this loop to compare prompts.py variants and pick the best one.


## Summary

Validation harness complete:
- Gold standards loaded from YAML → canned_questions table ✓
- Validation runs stored in validation_runs table ✓
- Cosine similarity scorer (sentence-transformers) ✓
- Pass/fail table with threshold=0.75 ✓
- Score distribution histogram ✓
- Prompt comparison loop demo ✓

**System is now fully built.** See the End-to-End Verification checklist in the build plan.